In [1]:
%cd ../analysis/6.Update_WY_domain/

/var2/Works/junhyeong/RXLR_landscape/analysis/6.Update_WY_domain


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [12]:
import pandas as pd
from Bio import SearchIO

In [3]:
df = pd.read_csv("../3.Clustering/WY_containing_cluster_entries.tsv", sep = "\t")

In [4]:
df

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl
0,Hyaar1_13656,Hyaara,0.0,0.0,no (L)WY,AHTSAKEDQEGRMIPGSEFFEKAQPVLHNLSINMPNLDEIKANAAS...,-1,1
1,RAW21624,Phycac,4.0,4.0,LWY,AGLSVPVAEKLKTWFTSSKVTPKRLQSWLNKGKSAETVFTRMQLTK...,1,1
2,RAW21590,Phycac,3.0,2.0,LWY,GVPTSSIENFKSLFTSLKISDETLQRWLKNGKTADKLFIRLKLGSA...,1,1
3,RAW40891,Phycac,1.0,0.0,WY,GITVTGIENLVKSSSKKAQLKLWLTQKKSTDATFKLLNLERRRNFI...,1,1
4,RAW29717,Phycac,4.0,4.0,LWY,KLGISLSGLEQAASGTKSWAAKLIQKLQLKWWQLRKKTPNDVFTKL...,1,1
...,...,...,...,...,...,...,...,...
1758,Phyra86034,Phyram,2.0,1.0,LWY,GITLPSSATEKIAKLAMSSDEIARIQAKLDRVFKGIQLDKKNSYLF...,329,309
1759,Phyci1_28041,Phycin,0.0,0.0,no (L)WY,IGAVSTLTKSTSVNEKADQPTAMTSYVALVIGSTGAVGRDLVAELV...,-1,310
1760,Phylat_28083.3980,Phylat,1.0,0.0,WY,RLVNIDLILKDAAISVKKATAWKAQFLWWKTFNKKPFPLAQELGIS...,-1,310
1761,OWZ10857,Phymeg,0.0,0.0,no (L)WY,VLAARCGCNNSLAMWRQRPSLFMLSKSLRPARLLRPGARSFALGQH...,390,324


In [5]:
with open("WY_containing_cluster_entries.fasta", "w") as f:
    for idx in df.index:
        entry = df.loc[idx, "entry"]
        seq = df.loc[idx, "seq"]

        f.write(f">{entry}\n{seq}\n")

In [6]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair WY_containing_cluster_entries.fasta > WY_containing_cluster_entries.align.fasta

In [7]:
!conda run -n homology-env esl-reformat -o WY_containing_cluster_entries.align.sto stockholm WY_containing_cluster_entries.align.fasta

In [8]:
!conda run -n homology-env hmmbuild WY_containing_cluster_entries..hmm WY_containing_cluster_entries.align.sto

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             WY_containing_cluster_entries.align.sto
# output HMM file:                  WY_containing_cluster_entries.hmm
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     WY_containing_cluster_entries.align  1763  6664   164    44.59  0.590 

# CPU time: 0.67u 0.02s 00:00:00.69 Elapsed: 00:00:00.69



In [9]:
!conda run -n homology-env hmmsearch -o WY_containing_cluster_entries.results WY_containing_cluster_entries.hmm WY_containing_cluster_entries.fasta

In [13]:
newly_annotated_wy_domain = {}

for query in SearchIO.parse("./WY_containing_cluster_entries.results", "hmmer3-text"):
    for hit in query:
        if hit.bitscore > 0:
            newly_annotated_wy_domain[hit.id] = len(hit.hsps)

In [14]:
len(newly_annotated_wy_domain)

1431

In [15]:
df

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl
0,Hyaar1_13656,Hyaara,0.0,0.0,no (L)WY,AHTSAKEDQEGRMIPGSEFFEKAQPVLHNLSINMPNLDEIKANAAS...,-1,1
1,RAW21624,Phycac,4.0,4.0,LWY,AGLSVPVAEKLKTWFTSSKVTPKRLQSWLNKGKSAETVFTRMQLTK...,1,1
2,RAW21590,Phycac,3.0,2.0,LWY,GVPTSSIENFKSLFTSLKISDETLQRWLKNGKTADKLFIRLKLGSA...,1,1
3,RAW40891,Phycac,1.0,0.0,WY,GITVTGIENLVKSSSKKAQLKLWLTQKKSTDATFKLLNLERRRNFI...,1,1
4,RAW29717,Phycac,4.0,4.0,LWY,KLGISLSGLEQAASGTKSWAAKLIQKLQLKWWQLRKKTPNDVFTKL...,1,1
...,...,...,...,...,...,...,...,...
1758,Phyra86034,Phyram,2.0,1.0,LWY,GITLPSSATEKIAKLAMSSDEIARIQAKLDRVFKGIQLDKKNSYLF...,329,309
1759,Phyci1_28041,Phycin,0.0,0.0,no (L)WY,IGAVSTLTKSTSVNEKADQPTAMTSYVALVIGSTGAVGRDLVAELV...,-1,310
1760,Phylat_28083.3980,Phylat,1.0,0.0,WY,RLVNIDLILKDAAISVKKATAWKAQFLWWKTFNKKPFPLAQELGIS...,-1,310
1761,OWZ10857,Phymeg,0.0,0.0,no (L)WY,VLAARCGCNNSLAMWRQRPSLFMLSKSLRPARLLRPGARSFALGQH...,390,324


In [16]:
newly_wy = []
newly_wy_count = []
for idx in df.index:
    if df.loc[idx, "entry"] in newly_annotated_wy_domain:
        newly_wy.append("WY")
        newly_wy_count.append(newly_annotated_wy_domain[df.loc[idx, "entry"]])

    else:
        newly_wy.append("no WY")
        newly_wy_count.append(0)

In [17]:
df["newly_annotated_wy"] = newly_wy

In [18]:
df["newly_annotated_wy_count"] = newly_wy_count

In [19]:
df

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl,newly_annotated_wy,newly_annotated_wy_count
0,Hyaar1_13656,Hyaara,0.0,0.0,no (L)WY,AHTSAKEDQEGRMIPGSEFFEKAQPVLHNLSINMPNLDEIKANAAS...,-1,1,no WY,0
1,RAW21624,Phycac,4.0,4.0,LWY,AGLSVPVAEKLKTWFTSSKVTPKRLQSWLNKGKSAETVFTRMQLTK...,1,1,WY,3
2,RAW21590,Phycac,3.0,2.0,LWY,GVPTSSIENFKSLFTSLKISDETLQRWLKNGKTADKLFIRLKLGSA...,1,1,WY,2
3,RAW40891,Phycac,1.0,0.0,WY,GITVTGIENLVKSSSKKAQLKLWLTQKKSTDATFKLLNLERRRNFI...,1,1,WY,1
4,RAW29717,Phycac,4.0,4.0,LWY,KLGISLSGLEQAASGTKSWAAKLIQKLQLKWWQLRKKTPNDVFTKL...,1,1,WY,3
...,...,...,...,...,...,...,...,...,...,...
1758,Phyra86034,Phyram,2.0,1.0,LWY,GITLPSSATEKIAKLAMSSDEIARIQAKLDRVFKGIQLDKKNSYLF...,329,309,WY,1
1759,Phyci1_28041,Phycin,0.0,0.0,no (L)WY,IGAVSTLTKSTSVNEKADQPTAMTSYVALVIGSTGAVGRDLVAELV...,-1,310,no WY,0
1760,Phylat_28083.3980,Phylat,1.0,0.0,WY,RLVNIDLILKDAAISVKKATAWKAQFLWWKTFNKKPFPLAQELGIS...,-1,310,WY,1
1761,OWZ10857,Phymeg,0.0,0.0,no (L)WY,VLAARCGCNNSLAMWRQRPSLFMLSKSLRPARLLRPGARSFALGQH...,390,324,no WY,0


In [21]:
df.to_csv("WY_updated.tsv", sep  = "\t", index = False)

# Wy only

In [1]:
%cd ../analysis/6.Update_WY_domain/

/var2/Works/junhyeong/RXLR_landscape/analysis/6.Update_WY_domain


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import pandas as pd
from Bio import SearchIO

In [3]:
df = pd.read_csv("../3.Clustering/WY_containing_cluster_entries.tsv", sep = "\t")

In [4]:
df

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl
0,Hyaar1_13656,Hyaara,0.0,0.0,no (L)WY,AHTSAKEDQEGRMIPGSEFFEKAQPVLHNLSINMPNLDEIKANAAS...,-1,1
1,RAW21624,Phycac,4.0,4.0,LWY,AGLSVPVAEKLKTWFTSSKVTPKRLQSWLNKGKSAETVFTRMQLTK...,1,1
2,RAW21590,Phycac,3.0,2.0,LWY,GVPTSSIENFKSLFTSLKISDETLQRWLKNGKTADKLFIRLKLGSA...,1,1
3,RAW40891,Phycac,1.0,0.0,WY,GITVTGIENLVKSSSKKAQLKLWLTQKKSTDATFKLLNLERRRNFI...,1,1
4,RAW29717,Phycac,4.0,4.0,LWY,KLGISLSGLEQAASGTKSWAAKLIQKLQLKWWQLRKKTPNDVFTKL...,1,1
...,...,...,...,...,...,...,...,...
1758,Phyra86034,Phyram,2.0,1.0,LWY,GITLPSSATEKIAKLAMSSDEIARIQAKLDRVFKGIQLDKKNSYLF...,329,309
1759,Phyci1_28041,Phycin,0.0,0.0,no (L)WY,IGAVSTLTKSTSVNEKADQPTAMTSYVALVIGSTGAVGRDLVAELV...,-1,310
1760,Phylat_28083.3980,Phylat,1.0,0.0,WY,RLVNIDLILKDAAISVKKATAWKAQFLWWKTFNKKPFPLAQELGIS...,-1,310
1761,OWZ10857,Phymeg,0.0,0.0,no (L)WY,VLAARCGCNNSLAMWRQRPSLFMLSKSLRPARLLRPGARSFALGQH...,390,324


In [5]:
with open("WY_cluster_entries.fasta", "w") as f:
    for idx in df.index:

        if df.loc[idx, "str_cl"] not in [3, 4, 5, 10, 11, 12, 13, 17, 18, 19, 22, 25, 27]:
            continue
            
        entry = df.loc[idx, "entry"]
        seq = df.loc[idx, "seq"]

        f.write(f">{entry}\n{seq}\n")

In [6]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair WY_cluster_entries.fasta > WY_cluster_entries.align.fasta

In [7]:
!conda run -n homology-env esl-reformat -o WY_cluster_entries.align.sto stockholm WY_cluster_entries.align.fasta

In [8]:
!conda run -n homology-env hmmbuild WY_cluster_entries.hmm WY_cluster_entries.align.sto

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             WY_cluster_entries.align.sto
# output HMM file:                  WY_cluster_entries.hmm
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     WY_cluster_entries.align   671  1174    98    53.60  0.590 

# CPU time: 0.20u 0.00s 00:00:00.20 Elapsed: 00:00:00.20



In [9]:
!conda run -n homology-env hmmsearch -o WY_cluster_entries.results WY_cluster_entries.hmm WY_cluster_entries.fasta

In [11]:
newly_annotated_wy_domain = {}

for query in SearchIO.parse("./WY_cluster_entries.results", "hmmer3-text"):
    for hit in query:
        if hit.bitscore > 0:
            newly_annotated_wy_domain[hit.id] = len(hit.hsps)

In [15]:
df2 = df[[str_cl in [3, 4, 5, 10, 11, 12, 13, 17, 18, 19, 22, 25, 27] for str_cl in df["str_cl"]]]

In [16]:
newly_wy = []
newly_wy_count = []
for idx in df2.index:
    if df2.loc[idx, "entry"] in newly_annotated_wy_domain:
        newly_wy.append("WY")
        newly_wy_count.append(newly_annotated_wy_domain[df2.loc[idx, "entry"]])

    else:
        newly_wy.append("no WY")
        newly_wy_count.append(0)

In [17]:
df2

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl
300,Phyker_46568.2426,Phyker,2.0,0.0,WY,NSVDDEERGNSVSQKWLNQFSKWHRRNEKADDVYARYALEPVVRKA...,-1,3
301,OWY99081,Phymeg,0.0,0.0,no (L)WY,DILSTAATKLAGKVKTNINKLLHWKLTKMEKMYRDQISPGDLAMKY...,-1,3
302,OWZ03877,Phymeg,0.0,0.0,no (L)WY,GFSTEDAKAFITKLVKMKTISDLWKVDDLTKLDKILDAAGDHMESK...,6,3
303,Phyci1_283183,Phycin,1.0,0.0,WY,TSDTTVTEDTKPVAPNRKFIEHKLQKALSNPKKTQRLYKTWYKKGY...,10,3
304,Phylat_28083.10092,Phylat,1.0,0.0,WY,TAPHEKMTESEKEAGPVFMEHKLVKALTNPKKTRRLYQSWYKRGIT...,10,3
...,...,...,...,...,...,...,...,...
1266,OWZ02970,Phymeg,1.0,0.0,WY,GTGFKDAVAKVVEKVRVNALKVKIPYWIVTGKDGPQVQKALGMAGV...,75,27
1267,OWZ02171,Phymeg,1.0,0.0,WY,GIRDAVEKGRITALKAKIPLWILQRKSQSDVQELLGMAGVYPLINH...,75,27
1268,OWY99227,Phymeg,1.0,0.0,WY,GIRDAVEKARITALKAKIPLWIVQRKSQSDVQELLGMAGVYPLINH...,75,27
1269,XP_009516815.1,Physoj,1.0,0.0,WY,GGGGYNTKMLVDLAKSKNNKLPGIIAKLSKSGQKTVINTWRKQSKN...,187,27


In [19]:
df2["newly_annotated_wy"] = newly_wy

/tmp/ipykernel_1928404/135271823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["newly_annotated_wy"] = newly_wy


In [20]:
df2["newly_annotated_wy_count"] = newly_wy_count

/tmp/ipykernel_1928404/1587459324.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["newly_annotated_wy_count"] = newly_wy_count


In [22]:
df2[df2["type"] == "WY"]

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl,newly_annotated_wy,newly_annotated_wy_count
300,Phyker_46568.2426,Phyker,2.0,0.0,WY,NSVDDEERGNSVSQKWLNQFSKWHRRNEKADDVYARYALEPVVRKA...,-1,3,WY,1
303,Phyci1_283183,Phycin,1.0,0.0,WY,TSDTTVTEDTKPVAPNRKFIEHKLQKALSNPKKTQRLYKTWYKKGY...,10,3,WY,1
304,Phylat_28083.10092,Phylat,1.0,0.0,WY,TAPHEKMTESEKEAGPVFMEHKLVKALTNPKKTRRLYQSWYKRGIT...,10,3,WY,1
307,XP_009528277.1,Physoj,1.0,0.0,WY,AADTKTADDTSTINPKGEFIEHKLQRALTNPKKTQRLYKTWYNKGY...,10,3,WY,1
315,RAW27676,Phycac,1.0,0.0,WY,FSLIQTSNTPRYYWWFQNEMTPRDVRKELGLRSDSIKLVKRSIYKG...,15,3,WY,1
...,...,...,...,...,...,...,...,...,...,...
1266,OWZ02970,Phymeg,1.0,0.0,WY,GTGFKDAVAKVVEKVRVNALKVKIPYWIVTGKDGPQVQKALGMAGV...,75,27,no WY,0
1267,OWZ02171,Phymeg,1.0,0.0,WY,GIRDAVEKGRITALKAKIPLWILQRKSQSDVQELLGMAGVYPLINH...,75,27,WY,1
1268,OWY99227,Phymeg,1.0,0.0,WY,GIRDAVEKARITALKAKIPLWIVQRKSQSDVQELLGMAGVYPLINH...,75,27,WY,1
1269,XP_009516815.1,Physoj,1.0,0.0,WY,GGGGYNTKMLVDLAKSKNNKLPGIIAKLSKSGQKTVINTWRKQSKN...,187,27,WY,1


In [23]:
df2[df2["newly_annotated_wy"] == "WY"]

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl,newly_annotated_wy,newly_annotated_wy_count
300,Phyker_46568.2426,Phyker,2.0,0.0,WY,NSVDDEERGNSVSQKWLNQFSKWHRRNEKADDVYARYALEPVVRKA...,-1,3,WY,1
301,OWY99081,Phymeg,0.0,0.0,no (L)WY,DILSTAATKLAGKVKTNINKLLHWKLTKMEKMYRDQISPGDLAMKY...,-1,3,WY,1
302,OWZ03877,Phymeg,0.0,0.0,no (L)WY,GFSTEDAKAFITKLVKMKTISDLWKVDDLTKLDKILDAAGDHMESK...,6,3,WY,1
303,Phyci1_283183,Phycin,1.0,0.0,WY,TSDTTVTEDTKPVAPNRKFIEHKLQKALSNPKKTQRLYKTWYKKGY...,10,3,WY,1
304,Phylat_28083.10092,Phylat,1.0,0.0,WY,TAPHEKMTESEKEAGPVFMEHKLVKALTNPKKTRRLYQSWYKRGIT...,10,3,WY,1
...,...,...,...,...,...,...,...,...,...,...
1265,Phyra83593,Phyram,3.0,0.0,WY,GILTGVKMAVKTTYWAQMQKSEDFVKKALKLDGLTKGAMKANPKYK...,62,27,WY,3
1267,OWZ02171,Phymeg,1.0,0.0,WY,GIRDAVEKGRITALKAKIPLWILQRKSQSDVQELLGMAGVYPLINH...,75,27,WY,1
1268,OWY99227,Phymeg,1.0,0.0,WY,GIRDAVEKARITALKAKIPLWIVQRKSQSDVQELLGMAGVYPLINH...,75,27,WY,1
1269,XP_009516815.1,Physoj,1.0,0.0,WY,GGGGYNTKMLVDLAKSKNNKLPGIIAKLSKSGQKTVINTWRKQSKN...,187,27,WY,1


In [24]:
df2.to_csv("WY_updated.tsv", sep  = "\t", index = False)